## Regression Predictions of Exam Scores

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

import matplotlib.pyplot as plt

In [8]:
# Example data
np.random.seed(42)
Student = pd.read_csv("student_performance_factors.csv")

X = Student.iloc[:, :-1]
scores = Student.loc[:, "Exam_Score"]



categorical_cols = ["Parental_Involvement", "Access_to_Resources","Extracurricular_Activities","Motivation_Level",
"Internet_Access", "Family_Income" , "Teacher_Quality"  , "School_Type", "Peer_Influence",    
"Learning_Disabilities", "Parental_Education_Level",   "Distance_from_Home" , "Gender"]

numeric = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Tutoring_Sessions","Physical_Activity"]


dummies_drop = pd.get_dummies(X[categorical_cols], drop_first=True)

Xdummies = pd.concat([X[numeric], dummies_drop], axis=1)

poly = PolynomialFeatures(degree=2, include_bias=True)

X_interactions = poly.fit_transform(Xdummies)

# Get feature names
feature_names = poly.get_feature_names_out(Xdummies.columns)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_interactions, scores, test_size=0.2, random_state=42
)

# Scale features (important for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create LassoCV model
lasso_cv = LassoCV(
    cv=5,                  # 5-fold cross-validation
    random_state=42,
    alphas=None,           # Auto-generate alpha values
    max_iter=10000,
    n_alphas=100           # Number of alphas to try
)

# Fit the model
lasso_cv.fit(X_train_scaled, y_train)

# Print results
print(f"Best alpha: {lasso_cv.alpha_:.6f}")
print(f"R² Score: {lasso_cv.score(X_test_scaled, y_test):.4f}")

# Make predictions
y_pred = lasso_cv.predict(X_test_scaled)
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1622: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1641: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.027009
R² Score: 0.7644
RMSE: 1.8249


In [9]:
np.sum(lasso_cv.coef_ != 0)

np.int64(133)